In [ ]:
# ConnectX Cell Swarm Starter Agent
# Educational baseline agent for Kaggle ConnectX.

from kaggle_environments import make

env = make("connectx", debug=True)

In [ ]:
# Tactical wrapper around the Cell Swarm ConnectX agent. Pure Python.

class _Obj:
    pass


def _as_obj(value):
    if isinstance(value, dict):
        obj = _Obj()
        for key, item in value.items():
            setattr(obj, key, item)
        return obj
    return value


def _dims(conf):
    return conf.rows, conf.columns, conf.inarow


def _center_order(columns):
    center = columns // 2
    return sorted(range(columns), key=lambda col: (abs(col - center), col))


def _valid_actions(board, rows, columns):
    return [col for col in _center_order(columns) if board[col] == 0]


def _drop(board, col, mark, rows, columns):
    for row in range(rows - 1, -1, -1):
        idx = row * columns + col
        if board[idx] == 0:
            board[idx] = mark
            return row, idx
    return -1, -1


def _undo(board, idx):
    board[idx] = 0


def _won_from(board, row, col, mark, rows, columns, inarow):
    for dr, dc in ((0, 1), (1, 0), (1, 1), (1, -1)):
        count = 1
        r, c = row + dr, col + dc
        while 0 <= r < rows and 0 <= c < columns and board[r * columns + c] == mark:
            count += 1
            r += dr
            c += dc
        r, c = row - dr, col - dc
        while 0 <= r < rows and 0 <= c < columns and board[r * columns + c] == mark:
            count += 1
            r -= dr
            c -= dc
        if count >= inarow:
            return True
    return False


def _find_win(board, mark, rows, columns, inarow):
    for col in _valid_actions(board, rows, columns):
        row, idx = _drop(board, col, mark, rows, columns)
        won = _won_from(board, row, col, mark, rows, columns, inarow)
        _undo(board, idx)
        if won:
            return col
    return None


def _count_wins(board, mark, rows, columns, inarow):
    count = 0
    for col in _valid_actions(board, rows, columns):
        row, idx = _drop(board, col, mark, rows, columns)
        won = _won_from(board, row, col, mark, rows, columns, inarow)
        _undo(board, idx)
        if won:
            count += 1
    return count


def _allows_opp_win(board, col, mark, opp_mark, rows, columns, inarow):
    row, idx = _drop(board, col, mark, rows, columns)
    if idx < 0:
        return True
    allows = _find_win(board, opp_mark, rows, columns, inarow) is not None
    _undo(board, idx)
    return allows


def _safe_fallback(board, mark, opp_mark, rows, columns, inarow):
    best_col = None
    best_score = None
    for col in _valid_actions(board, rows, columns):
        if _allows_opp_win(board, col, mark, opp_mark, rows, columns, inarow):
            continue
        score = _score_after_move(board, col, mark, opp_mark, rows, columns, inarow)
        if best_score is None or score > best_score:
            best_col = col
            best_score = score
    if best_col is not None:
        return best_col
    actions = _valid_actions(board, rows, columns)
    return actions[0] if actions else 0


def _count_windows(board, stones, mark, rows, columns, inarow):
    windows = 0
    for r in range(rows):
        for c in range(columns - inarow + 1):
            window = [board[r * columns + c + i] for i in range(inarow)]
            if window.count(mark) == stones and window.count(0) == inarow - stones:
                windows += 1

    for c in range(columns):
        for r in range(rows - inarow + 1):
            window = [board[(r + i) * columns + c] for i in range(inarow)]
            if window.count(mark) == stones and window.count(0) == inarow - stones:
                windows += 1

    for r in range(rows - inarow + 1):
        for c in range(columns - inarow + 1):
            window = [board[(r + i) * columns + c + i] for i in range(inarow)]
            if window.count(mark) == stones and window.count(0) == inarow - stones:
                windows += 1

    for r in range(inarow - 1, rows):
        for c in range(columns - inarow + 1):
            window = [board[(r - i) * columns + c + i] for i in range(inarow)]
            if window.count(mark) == stones and window.count(0) == inarow - stones:
                windows += 1

    return windows


def _heuristic(board, mark, opp_mark, rows, columns, inarow):
    center = columns // 2
    center_score = 80 * sum(1 for r in range(rows) if board[r * columns + center] == mark)
    own_twos = _count_windows(board, 2, mark, rows, columns, inarow)
    own_threes = _count_windows(board, 3, mark, rows, columns, inarow)
    own_fours = _count_windows(board, 4, mark, rows, columns, inarow)
    opp_twos = _count_windows(board, 2, opp_mark, rows, columns, inarow)
    opp_threes = _count_windows(board, 3, opp_mark, rows, columns, inarow)
    opp_fours = _count_windows(board, 4, opp_mark, rows, columns, inarow)
    return (
        center_score
        + 2 * own_twos
        + 30 * own_threes
        + 1000000 * own_fours
        - 300 * opp_threes
        - 1000000 * opp_fours
    )


def _score_after_move(board, col, mark, opp_mark, rows, columns, inarow):
    row, idx = _drop(board, col, mark, rows, columns)
    if idx < 0:
        return -10 ** 12
    score = _heuristic(board, mark, opp_mark, rows, columns, inarow)
    _undo(board, idx)
    return score


def _lookahead_score(board, depth, turn_mark, root_mark, opp_mark, rows, columns, inarow, alpha, beta):
    other_mark = 1 if turn_mark == 2 else 2
    actions = _valid_actions(board, rows, columns)
    if not actions:
        return 0

    immediate = _find_win(board, turn_mark, rows, columns, inarow)
    if immediate is not None:
        if turn_mark == root_mark:
            return 10 ** 10 + depth
        return -(10 ** 10) - depth

    if depth == 0:
        return _heuristic(board, root_mark, opp_mark, rows, columns, inarow)

    if turn_mark == root_mark:
        value = -(10 ** 12)
        for col in actions:
            row, idx = _drop(board, col, turn_mark, rows, columns)
            child = _lookahead_score(board, depth - 1, other_mark, root_mark, opp_mark, rows, columns, inarow, alpha, beta)
            _undo(board, idx)
            if child > value:
                value = child
            alpha = max(alpha, value)
            if alpha >= beta:
                break
        return value

    value = 10 ** 12
    for col in actions:
        row, idx = _drop(board, col, turn_mark, rows, columns)
        child = _lookahead_score(board, depth - 1, other_mark, root_mark, opp_mark, rows, columns, inarow, alpha, beta)
        _undo(board, idx)
        if child < value:
            value = child
        beta = min(beta, value)
        if alpha >= beta:
            break
    return value


def _lookahead_action(board, mark, opp_mark, rows, columns, inarow, depth):
    best_col = None
    best_score = None
    alpha = -(10 ** 12)
    beta = 10 ** 12
    for col in _valid_actions(board, rows, columns):
        row, idx = _drop(board, col, mark, rows, columns)
        if _won_from(board, row, col, mark, rows, columns, inarow):
            score = 10 ** 10 + depth
        else:
            score = _lookahead_score(board, depth - 1, opp_mark, mark, opp_mark, rows, columns, inarow, alpha, beta)
        _undo(board, idx)
        if best_score is None or score > best_score:
            best_score = score
            best_col = col
        alpha = max(alpha, best_score)
    return best_col, best_score


def _best_tactical_action(board, mark, opp_mark, rows, columns, inarow):
    actions = _valid_actions(board, rows, columns)
    safe = []
    for col in actions:
        if not _allows_opp_win(board, col, mark, opp_mark, rows, columns, inarow):
            safe.append(col)
    if not safe:
        safe = actions

    # Prefer moves that create multiple immediate winning threats.
    for col in safe:
        row, idx = _drop(board, col, mark, rows, columns)
        wins = _count_wins(board, mark, rows, columns, inarow)
        _undo(board, idx)
        if wins >= 2:
            return col

    best_col = safe[0] if safe else 0
    best_opp_wins = None
    best_score = None
    center = columns // 2
    for col in safe:
        row, idx = _drop(board, col, mark, rows, columns)
        opp_wins = _count_wins(board, opp_mark, rows, columns, inarow)
        score = _heuristic(board, mark, opp_mark, rows, columns, inarow)
        _undo(board, idx)
        center_distance = abs(col - center)
        if (
            best_opp_wins is None
            or opp_wins < best_opp_wins
            or (opp_wins == best_opp_wins and score > best_score)
            or (
                opp_wins == best_opp_wins
                and score == best_score
                and center_distance < abs(best_col - center)
            )
        ):
            best_col = col
            best_opp_wins = opp_wins
            best_score = score
    return best_col


def cell_swarm(obs, conf):
    def evaluate_cell(cell):
        """ evaluate qualities of the cell """
        cell = get_patterns(cell)
        cell = calculate_points(cell)
        for i in range(1, conf.rows):
            cell = explore_cell_above(cell, i)
        return cell
    
    def get_patterns(cell):
        """ get swarm and opponent's patterns of each axis of the cell """
        ne = get_pattern(cell["x"], lambda z : z + 1, cell["y"], lambda z : z - 1, conf.inarow)
        sw = get_pattern(cell["x"], lambda z : z - 1, cell["y"], lambda z : z + 1, conf.inarow)[::-1]
        cell["swarm_patterns"]["NE_SW"] = sw + [{"mark": swarm_mark}] + ne
        cell["opp_patterns"]["NE_SW"] = sw + [{"mark": opp_mark}] + ne
        e = get_pattern(cell["x"], lambda z : z + 1, cell["y"], lambda z : z, conf.inarow)
        w = get_pattern(cell["x"], lambda z : z - 1, cell["y"], lambda z : z, conf.inarow)[::-1]
        cell["swarm_patterns"]["E_W"] = w + [{"mark": swarm_mark}] + e
        cell["opp_patterns"]["E_W"] = w + [{"mark": opp_mark}] + e
        se = get_pattern(cell["x"], lambda z : z + 1, cell["y"], lambda z : z + 1, conf.inarow)
        nw = get_pattern(cell["x"], lambda z : z - 1, cell["y"], lambda z : z - 1, conf.inarow)[::-1]
        cell["swarm_patterns"]["SE_NW"] = nw + [{"mark": swarm_mark}] + se
        cell["opp_patterns"]["SE_NW"] = nw + [{"mark": opp_mark}] + se
        s = get_pattern(cell["x"], lambda z : z, cell["y"], lambda z : z + 1, conf.inarow)
        n = get_pattern(cell["x"], lambda z : z, cell["y"], lambda z : z - 1, conf.inarow)[::-1]
        cell["swarm_patterns"]["S_N"] = n + [{"mark": swarm_mark}] + s
        cell["opp_patterns"]["S_N"] = n + [{"mark": opp_mark}] + s
        return cell
        
    def get_pattern(x, x_fun, y, y_fun, cells_remained):
        """ get pattern of marks in direction """
        pattern = []
        x = x_fun(x)
        y = y_fun(y)
        # if cell is inside swarm's borders
        if y >= 0 and y < conf.rows and x >= 0 and x < conf.columns:
            pattern.append({
                "mark": swarm[x][y]["mark"]
            })
            # amount of cells to explore in this direction
            cells_remained -= 1
            if cells_remained > 1:
                pattern.extend(get_pattern(x, x_fun, y, y_fun, cells_remained))
        return pattern
    
    def calculate_points(cell):
        """ calculate amounts of swarm's and opponent's correct patterns and add them to cell's points """
        for i in range(conf.inarow - 1):
            # inarow = amount of marks in pattern to consider that pattern as correct
            inarow = conf.inarow - i
            swarm_points = 0
            opp_points = 0
            # calculate swarm's points and depth
            swarm_points = evaluate_pattern(swarm_points, cell["swarm_patterns"]["E_W"], swarm_mark, inarow)
            swarm_points = evaluate_pattern(swarm_points, cell["swarm_patterns"]["NE_SW"], swarm_mark, inarow)
            swarm_points = evaluate_pattern(swarm_points, cell["swarm_patterns"]["SE_NW"], swarm_mark, inarow)
            swarm_points = evaluate_pattern(swarm_points, cell["swarm_patterns"]["S_N"], swarm_mark, inarow)
            # calculate opponent's points and depth
            opp_points = evaluate_pattern(opp_points, cell["opp_patterns"]["E_W"], opp_mark, inarow)
            opp_points = evaluate_pattern(opp_points, cell["opp_patterns"]["NE_SW"], opp_mark, inarow)
            opp_points = evaluate_pattern(opp_points, cell["opp_patterns"]["SE_NW"], opp_mark, inarow)
            opp_points = evaluate_pattern(opp_points, cell["opp_patterns"]["S_N"], opp_mark, inarow)
            # if more than one mark required for victory
            if i > 0:
                # swarm_mark or opp_mark priority
                if swarm_points > opp_points:
                    cell["points"].append(swarm_points)
                    cell["points"].append(opp_points)
                else:
                    cell["points"].append(opp_points)
                    cell["points"].append(swarm_points)
            else:
                cell["points"].append(swarm_points)
                cell["points"].append(opp_points)
        return cell
                    
    def evaluate_pattern(points, pattern, mark, inarow):
        """ get amount of points, if pattern has required amounts of marks and zeros """
        # saving enough cells for required amounts of marks and zeros
        for i in range(len(pattern) - (conf.inarow - 1)):
            marks = 0
            zeros = 0
            # check part of pattern for required amounts of marks and zeros
            for j in range(conf.inarow):
                if pattern[i + j]["mark"] == mark:
                    marks += 1
                elif pattern[i + j]["mark"] == 0:
                    zeros += 1
            if marks >= inarow and (marks + zeros) == conf.inarow:
                return points + 1
        return points
    
    def explore_cell_above(cell, i):
        """ add positive or negative points from cell above (if it exists) to points of current cell """
        if (cell["y"] - i) >= 0:
            cell_above = swarm[cell["x"]][cell["y"] - i]
            cell_above = get_patterns(cell_above)
            cell_above = calculate_points(cell_above)
            # points will be positive or negative
            n = -1 if i & 1 else 1
            # if it is first cell above
            if i == 1:
                # add first 4 points of cell_above["points"] to cell["points"]
                cell["points"][2:2] = [n * cell_above["points"][1], n * cell_above["points"][0]]
                # if it is not potential "seven" pattern in cell and cell_above has more points
                if abs(cell["points"][4]) < 2 and abs(cell["points"][4]) < cell_above["points"][2]:
                    cell["points"][4:4] = [n * cell_above["points"][2]]
                    # if it is not potential "seven" pattern in cell and cell_above has more points
                    if abs(cell["points"][5]) < 2 and abs(cell["points"][5]) < cell_above["points"][3]:
                        cell["points"][5:5] = [n * cell_above["points"][3]]
                    else:
                        cell["points"][7:7] = [n * cell_above["points"][3]]
                else:
                    cell["points"][6:6] = [n * cell_above["points"][2], n * cell_above["points"][3]]
                cell["points"].append(n * cell_above["points"][4])
                cell["points"].append(n * cell_above["points"][5])
            else:
                cell["points"].extend(map(lambda z : z * n, cell_above["points"]))
        else:
            cell["points"].extend([0, 0, 0, 0, 0, 0])
        return cell
    
    def choose_best_cell(best_cell, current_cell):
        """ compare two cells and return the best one """
        if best_cell is not None:
            for i in range(len(best_cell["points"])):
                # compare amounts of points of two cells
                if best_cell["points"][i] < current_cell["points"][i]:
                    best_cell = current_cell
                    break
                if best_cell["points"][i] > current_cell["points"][i]:
                    break
                # if ["points"][i] of cells are equal, compare distance to swarm's center of each cell
                if best_cell["points"][i] > 0:
                    if best_cell["distance_to_center"] > current_cell["distance_to_center"]:
                        best_cell = current_cell
                        break
                    if best_cell["distance_to_center"] < current_cell["distance_to_center"]:
                        break
        else:
            best_cell = current_cell
        return best_cell
        
    
###############################################################################
    # define swarm's and opponent's marks
    swarm_mark = obs.mark
    opp_mark = 2 if swarm_mark == 1 else 1
    # define swarm's center
    swarm_center_horizontal = conf.columns // 2
    swarm_center_vertical = conf.rows // 2
    
    # define swarm as two dimensional array of cells
    swarm = []
    for column in range(conf.columns):
        swarm.append([])
        for row in range(conf.rows):
            cell = {
                        "x": column,
                        "y": row,
                        "mark": obs.board[conf.columns * row + column],
                        "swarm_patterns": {},
                        "opp_patterns": {},
                        "distance_to_center": abs(row - swarm_center_vertical) + abs(column - swarm_center_horizontal),
                        "points": []
                    }
            swarm[column].append(cell)
    
    best_cell = None
    # start searching for best_cell from swarm center
    x = swarm_center_horizontal
    # shift to right or left from swarm center
    shift = 0
    
    # searching for best_cell
    while x >= 0 and x < conf.columns:
        # find first empty cell starting from bottom of the column
        y = conf.rows - 1
        while y >= 0 and swarm[x][y]["mark"] != 0:
            y -= 1
        # if column is not full
        if y >= 0:
            # current cell evaluates its own qualities
            current_cell = evaluate_cell(swarm[x][y])
            # current cell compares itself against best cell
            best_cell = choose_best_cell(best_cell, current_cell)
                        
        # shift x to right or left from swarm center
        if shift >= 0:
            shift += 1
        shift *= -1
        x = swarm_center_horizontal + shift

    # return index of the best cell column
    return best_cell["x"]

def my_agent(observation, configuration):
    obs = _as_obj(observation)
    conf = _as_obj(configuration)
    rows, columns, inarow = _dims(conf)
    board = list(obs.board)
    mark = obs.mark
    opp_mark = 1 if mark == 2 else 2

    # Anti-center opening for the second player. Against Cell Swarm-style
    # agents, mirroring the center loses; a shoulder response wins locally.
    if mark == 2 and board.count(1) + board.count(2) == 1:
        center = columns // 2
        if board[(rows - 1) * columns + center] == opp_mark and board[2] == 0:
            return 2

    win = _find_win(board, mark, rows, columns, inarow)
    if win is not None:
        return win

    block = _find_win(board, opp_mark, rows, columns, inarow)
    if block is not None:
        return block

    tactical_action = _best_tactical_action(board, mark, opp_mark, rows, columns, inarow)
    lookahead_action, lookahead_score = _lookahead_action(board, mark, opp_mark, rows, columns, inarow, depth=3)
    if lookahead_action is not None and lookahead_score is not None and lookahead_score > -(10 ** 9):
        tactical_action = lookahead_action

    action = cell_swarm(obs, conf)
    if action is None or action < 0 or action >= columns or board[action] != 0:
        return tactical_action

    if _allows_opp_win(board, action, mark, opp_mark, rows, columns, inarow):
        return tactical_action

    row, idx = _drop(board, action, mark, rows, columns)
    action_opp_wins = _count_wins(board, opp_mark, rows, columns, inarow)
    _undo(board, idx)

    row, idx = _drop(board, tactical_action, mark, rows, columns)
    tactical_opp_wins = _count_wins(board, opp_mark, rows, columns, inarow)
    tactical_wins = _count_wins(board, mark, rows, columns, inarow)
    _undo(board, idx)

    if tactical_wins >= 2 or tactical_opp_wins < action_opp_wins:
        return tactical_action

    row, idx = _drop(board, action, mark, rows, columns)
    action_score = _lookahead_score(board, 2, opp_mark, mark, opp_mark, rows, columns, inarow, -(10 ** 12), 10 ** 12)
    _undo(board, idx)

    if lookahead_score is not None and lookahead_score > action_score:
        return tactical_action

    return action

In [ ]:
env.run([my_agent, "random"])


In [ ]:
from kaggle_environments import make, evaluate

env = make("connectx", debug=True)

results = evaluate(
    "connectx",
    [my_agent, "random"],
    num_episodes=10
)

results

In [ ]:
env.run([my_agent, "random"])

board = env.state[0].observation.board

for r in range(6):
    print(board[r*7:(r+1)*7])